# HW2 — Uniform Convergence and the Limits of Learning

**Statistical and Machine Learning — M2 IM / IMDS / MIAGE, Université de Haute-Alsace**

We study, by Monte-Carlo simulation, three ideas:

1. **Uniform convergence** — how the empirical risk $L_S(h)$ fluctuates around the true risk $L_D(h)$, and how this is controlled by the sample size $m$ and the hypothesis-class size $|H|$.
2. **No-Free-Lunch** — with completely random targets, a learner cannot predict labels of inputs it has never seen.
3. **Inductive bias** — restricting the target and the hypothesis class to a structured family (thresholds) makes generalization possible.

Only `numpy` and `matplotlib` are used.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)   # fixed seed -> reproducible experiments

## Part I — Simulating Uniform Convergence

We have a finite class $H=\{h_1,\dots,h_K\}$ where each $h_j$ has a fixed true risk $L_D(h_j)=p_j$.
On a sample of size $m$ the number of errors is $E_j\sim\text{Binomial}(m,p_j)$, so the empirical risk is
$$L_S(h_j)=\frac{E_j}{m}.$$
Simulating the binomial directly lets us study the fluctuations of $L_S$ around $L_D$.

### 1. Empirical risk as a random variable

For one hypothesis with $p=0.3$ we draw $L_S$ many times, for several sample sizes $m$, and look at its distribution.

In [ ]:
p  = 0.3                       # true risk of the chosen hypothesis
ms = [10, 50, 200, 1000]       # sample sizes
R  = 20000                     # Monte-Carlo repetitions

fig, axes = plt.subplots(1, len(ms), figsize=(16, 3), sharey=True)
for ax, m in zip(axes, ms):
    LS = rng.binomial(m, p, size=R) / m
    ax.hist(LS, bins=30, density=True, color="steelblue", alpha=.8)
    ax.axvline(p, color="red", lw=2, label="$L_D=p$")
    ax.axvline(LS.mean(), color="black", ls="--", label="mean $L_S$")
    ax.set_title(f"m={m}\nmean={LS.mean():.3f}, std={LS.std():.3f}")
    ax.set_xlabel("$L_S(h)$")
axes[0].set_ylabel("density"); axes[0].legend()
plt.tight_layout(); plt.show()

**Observation.** The empirical mean of $L_S$ matches $L_D=p$ for every $m$ (the estimator is unbiased).
As $m$ grows the histogram concentrates around $p$ and the standard deviation shrinks like
$\sqrt{p(1-p)/m}$ — larger samples give less variable empirical risks.

### 2. Deviation between empirical and true risk

For a fixed accuracy $\epsilon$ we estimate $\Pr[\,|L_S-L_D|>\epsilon\,]$ and compare it with Hoeffding's bound
$$\Pr[\,|L_S(h)-L_D(h)|>\epsilon\,]\le 2e^{-2m\epsilon^2}.$$

In [ ]:
p, eps = 0.3, 0.1
ms = np.arange(10, 400, 10)
R  = 20000

emp   = np.array([np.mean(np.abs(rng.binomial(m, p, R)/m - p) > eps) for m in ms])
bound = np.minimum(2*np.exp(-2*ms*eps**2), 1.0)

plt.figure(figsize=(7, 4))
plt.plot(ms, emp,   "o-", label="experimental $\\Pr[|L_S-L_D|>\\epsilon]$")
plt.plot(ms, bound, "s-", label="Hoeffding bound $2e^{-2m\\epsilon^2}$")
plt.xlabel("m"); plt.ylabel("probability"); plt.title(f"p={p}, eps={eps}")
plt.legend(); plt.tight_layout(); plt.show()

**Observation.** The experimental probability always stays below the Hoeffding bound, so the bound is *valid*.
It is also clearly **conservative**: the true deviation probability decays much faster than the bound,
because Hoeffding holds for the worst case $p=1/2$ while here $p=0.3$ gives smaller variance.

### 3. From one hypothesis to uniform convergence

Now we look at **all** hypotheses at once. For each sample we compute
$$\Delta_S=\max_{h\in H}|L_S(h)-L_D(h)|$$
and estimate $\Pr[\Delta_S>\epsilon]$, comparing with the finite-class union bound
$$\Pr\!\left[\sup_{h\in H}|L_S(h)-L_D(h)|>\epsilon\right]\le 2|H|e^{-2m\epsilon^2}.$$
(Following the simplified model, the $K$ empirical risks are simulated as independent binomials.)

In [ ]:
K   = 20
ps  = rng.uniform(0, 1, K)      # a true risk for each hypothesis
eps = 0.1
ms  = np.arange(10, 400, 10)
R   = 4000

emp = []
for m in ms:
    LS  = rng.binomial(m, ps, size=(R, K)) / m
    dev = np.max(np.abs(LS - ps), axis=1)
    emp.append(np.mean(dev > eps))
emp   = np.array(emp)
bound = np.minimum(2*K*np.exp(-2*ms*eps**2), 1.0)

plt.figure(figsize=(7, 4))
plt.plot(ms, emp,   "o-", label="experimental $\\Pr[\\Delta_S>\\epsilon]$")
plt.plot(ms, bound, "s-", label="union bound $2|H|e^{-2m\\epsilon^2}$")
plt.xlabel("m"); plt.ylabel("probability"); plt.title(f"|H|={K}, eps={eps}")
plt.legend(); plt.tight_layout(); plt.show()

**Observation.** Controlling the *maximum* deviation is harder than controlling a single hypothesis, so
$\Pr[\Delta_S>\epsilon]$ is larger than in Task 2 and needs a bigger $m$ to vanish. The union bound still
holds and is still conservative (the factor $|H|$ assumes the worst, independent case).

### 4. Effect of the size of the hypothesis class

We fix $\epsilon$ and repeat the uniform-convergence experiment for several class sizes $|H|$.

In [ ]:
Ks  = [5, 10, 50, 100, 500]
eps = 0.1
ms  = np.arange(10, 500, 20)
R   = 2000

plt.figure(figsize=(7, 4))
for K in Ks:
    ps  = rng.uniform(0, 1, K)
    emp = [np.mean(np.max(np.abs(rng.binomial(m, ps, size=(R, K))/m - ps), axis=1) > eps)
           for m in ms]
    plt.plot(ms, emp, "o-", ms=3, label=f"|H|={K}")
plt.xlabel("m"); plt.ylabel("$\\Pr[\\Delta_S>\\epsilon]$"); plt.title(f"eps={eps}")
plt.legend(); plt.tight_layout(); plt.show()

**Discussion.**
- Increasing $|H|$ shifts the curves to the right: a larger class makes uniform convergence harder.
- A larger hypothesis class therefore **requires more data** to reach the same confidence.
- The sufficient sample size grows like $\log|H|$ (from $2|H|e^{-2m\epsilon^2}\le\delta \Rightarrow m\ge\frac{\log(2|H|/\delta)}{2\epsilon^2}$).
  Going from 5 to 500 hypotheses multiplies $|H|$ by 100 but only adds $\log 100\approx 4.6$ worth of samples — the dependence is mild.

### 5. Experimental sample complexity

Fix $\epsilon$ and $\delta$. We find the smallest tested $m$ with $\Pr[\Delta_S>\epsilon]\le\delta$ and compare it
with the theoretical sufficient size
$$m\ge\frac{\log(2|H|/\delta)}{2\epsilon^2}.$$

In [ ]:
eps, delta, K = 0.1, 0.1, 50
ps = rng.uniform(0, 1, K)
ms = np.arange(10, 800, 10)
R  = 4000

emp = np.array([np.mean(np.max(np.abs(rng.binomial(m, ps, size=(R, K))/m - ps), axis=1) > eps)
                for m in ms])
ok       = np.where(emp <= delta)[0]
m_exp    = ms[ok[0]]
m_theory = np.log(2*K/delta) / (2*eps**2)

print(f"experimental smallest m : {m_exp}")
print(f"theoretical sufficient m: {m_theory:.0f}")

plt.figure(figsize=(7, 4))
plt.plot(ms, emp, "o-", ms=3, label="$\\Pr[\\Delta_S>\\epsilon]$")
plt.axhline(delta, color="gray", ls=":", label=f"delta={delta}")
plt.axvline(m_exp,    color="green", ls="--", label=f"experimental m={m_exp}")
plt.axvline(m_theory, color="red",   ls="--", label=f"theoretical m={m_theory:.0f}")
plt.xlabel("m"); plt.ylabel("probability"); plt.legend(); plt.tight_layout(); plt.show()

**Discussion.** The experimental sample size is **smaller** than the theoretical one. The formula is a
*sufficient* guarantee derived from a worst-case union bound, so it over-estimates the data actually needed;
it is not the exact minimum sample size.

## Part II — Simulating No-Free-Lunch

Domain $X=\{1,\dots,N\}$ with the uniform distribution. For each repetition we draw a **fresh random target**
$f(x)\sim\text{Bernoulli}(1/2)$ independently for every $x$. The learner memorizes labels it has seen and, by
default, predicts $0$ on inputs it has never seen.

### 6. Seen and unseen inputs

We estimate the expected fraction of inputs never observed and compare with
$$\Pr[x\text{ unseen}]=\left(1-\tfrac1N\right)^m,$$
which — by linearity of expectation over the $N$ points — is also the expected fraction of unseen inputs.

In [ ]:
N  = 100
ms = np.arange(1, 500, 10)
R  = 1500

emp = []
for m in ms:
    S    = rng.integers(0, N, size=(R, m))
    frac = [(N - len(np.unique(s))) / N for s in S]
    emp.append(np.mean(frac))
theory = (1 - 1/N)**ms

plt.figure(figsize=(7, 4))
plt.plot(ms, emp,    "o-", ms=3, label="experimental fraction unseen")
plt.plot(ms, theory, "-",  label="$(1-1/N)^m$")
plt.xlabel("m"); plt.ylabel("fraction unseen"); plt.title(f"N={N}")
plt.legend(); plt.tight_layout(); plt.show()

**Observation.** Simulation and theory coincide. Each point is unseen with probability $(1-1/N)^m$, and the
expected *count* of unseen points is $N(1-1/N)^m$, so the expected *fraction* is $(1-1/N)^m$.

### 7. Error on unseen inputs

The memorization learner is correct on seen inputs and guesses on unseen ones. Since unseen labels are
independent $\text{Bernoulli}(1/2)$, it is wrong on half of them, giving
$$\mathbb{E}[L_D(A(S))]=\tfrac12\left(1-\tfrac1N\right)^m.$$

In [ ]:
N  = 100
ms = np.arange(1, 500, 10)
R  = 1500

emp = []
for m in ms:
    risks = []
    for _ in range(R):
        f    = rng.integers(0, 2, N)       # random target
        S    = rng.integers(0, N, m)       # training inputs
        pred = np.zeros(N, int)            # default prediction 0
        pred[S] = f[S]                     # memorize seen labels
        risks.append(np.mean(pred != f))
    emp.append(np.mean(risks))
theory = 0.5 * (1 - 1/N)**ms

plt.figure(figsize=(7, 4))
plt.plot(ms, emp,    "o-", ms=3, label="experimental $E[L_D]$")
plt.plot(ms, theory, "-",  label="$\\frac{1}{2}(1-1/N)^m$")
plt.xlabel("m"); plt.ylabel("expected true risk"); plt.title(f"N={N}")
plt.legend(); plt.tight_layout(); plt.show()

**Observation.** The two curves agree. The risk never goes below the error made on the unseen part of the
domain, which stays random — memorization alone cannot generalize.

### 8. Changing the prediction on unseen inputs

We compare three rules for unseen inputs: always $0$, always $1$, or a random $\text{Bernoulli}(1/2)$ guess.

In [ ]:
N  = 100
ms = np.arange(1, 300, 20)
R  = 1500
res = {"predict 0": [], "predict 1": [], "random": []}

for m in ms:
    acc = {k: [] for k in res}
    for _ in range(R):
        f = rng.integers(0, 2, N)
        S = rng.integers(0, N, m)
        for k in res:
            pred = (np.zeros(N, int) if k == "predict 0" else
                    np.ones(N, int)  if k == "predict 1" else
                    rng.integers(0, 2, N))
            pred[S] = f[S]
            acc[k].append(np.mean(pred != f))
    for k in res:
        res[k].append(np.mean(acc[k]))

plt.figure(figsize=(7, 4))
for k, v in res.items():
    plt.plot(ms, v, "o-", ms=3, label=k)
plt.xlabel("m"); plt.ylabel("expected true risk"); plt.title("random targets")
plt.legend(); plt.tight_layout(); plt.show()

**Observation.** All three strategies give essentially the **same** expected risk. When the target is drawn
uniformly at random, no rule for the unseen part is better than any other — this is exactly the
**No-Free-Lunch** principle: without assumptions on $f$, the training sample tells us nothing about unseen
labels, so no learner can be universally preferable.

## Part III — Introducing Prior Knowledge

Now the target is a **threshold** $f_\theta(x)=\mathbb{1}[x>\theta]$ with $\theta\in\{1,\dots,N-1\}$, and the
learner's class is the matching family $H=\{h_t(x)=\mathbb{1}[x>t]\}$. We select $t$ by ERM (minimum training error).

### 9. Learning with a matching inductive bias

For several $m$ we draw a threshold target, learn $h_S$ by ERM over all thresholds, and measure the true risk
over the whole domain.

In [ ]:
def threshold_erm_risk(N, ms, R):
    X  = np.arange(1, N+1)
    ts = np.arange(1, N)                       # candidate thresholds
    out = []
    for m in ms:
        risks = []
        for _ in range(R):
            theta = rng.integers(1, N)         # true threshold in {1..N-1}
            xs = rng.integers(1, N+1, m)       # sample inputs
            ys = (xs > theta).astype(int)      # labels from the target
            preds = (xs[None, :] > ts[:, None]).astype(int)   # each candidate on the sample
            errs  = np.mean(preds != ys[None, :], axis=1)     # training errors
            t_hat = ts[np.argmin(errs)]                       # ERM choice
            risks.append(np.mean((X > t_hat) != (X > theta))) # true risk over X
        out.append(np.mean(risks))
    return np.array(out)

N  = 100
ms = np.arange(1, 200, 10)
R  = 800
risk_threshold = threshold_erm_risk(N, ms, R)

plt.figure(figsize=(7, 4))
plt.plot(ms, risk_threshold, "o-", ms=3, color="darkgreen", label="threshold ERM")
plt.xlabel("m"); plt.ylabel("expected true risk $E[L_D(h_S)]$"); plt.title(f"N={N}")
plt.legend(); plt.tight_layout(); plt.show()

**Observation.** The true risk decreases quickly toward $0$ as $m$ grows — unlike Part II, the learner
generalizes to inputs it never saw, because knowing the target is a threshold lets a few points pin down $\theta$.

### 10. Random targets vs. structured targets

We overlay the random-target risk (Part II, memorization) and the threshold-ERM risk (Part III).

In [ ]:
N  = 100
ms = np.arange(1, 200, 10)
R  = 800

# random targets (memorization, predict 0 on unseen)
risk_random = []
for m in ms:
    risks = []
    for _ in range(R):
        f = rng.integers(0, 2, N)
        S = rng.integers(0, N, m)
        pred = np.zeros(N, int); pred[S] = f[S]
        risks.append(np.mean(pred != f))
    risk_random.append(np.mean(risks))

risk_threshold = threshold_erm_risk(N, ms, R)   # defined in Task 9

plt.figure(figsize=(7, 4))
plt.plot(ms, risk_random,    "o-", ms=3, label="random target (memorization)")
plt.plot(ms, risk_threshold, "o-", ms=3, label="threshold target + threshold ERM")
plt.xlabel("m"); plt.ylabel("expected true risk"); plt.title(f"N={N}")
plt.legend(); plt.tight_layout(); plt.show()

**Discussion.** With random targets the risk stays near $\tfrac12(1-1/N)^m$ — high until almost every point is
seen. With a threshold target learned by threshold ERM the risk collapses after only a handful of samples.
The assumption *"the target is a threshold"* is a strong **inductive bias**: the learner no longer has to see a
point to label it — it only needs to locate $\theta$, which a small sample already does. This structure is
what makes generalization beyond the observed examples possible.

### 11. Final analysis

- **Concentration of $L_S$.** $L_S(h)=E/m$ with $E\sim\text{Binomial}(m,p)$ has standard deviation
  $\sqrt{p(1-p)/m}\to 0$, so empirical risk concentrates around the true risk as $m$ grows (Tasks 1–2).
- **Uniform convergence.** Guaranteeing good estimates for *every* hypothesis means controlling the maximum
  deviation $\Delta_S$, which is harder than controlling one hypothesis and needs more data (Task 3).
- **Role of $|H|$.** A larger class makes the maximum deviation more likely, so more data is required; the cost
  grows only like $\log|H|$ (Tasks 4–5).
- **Conservative bounds.** Hoeffding and the union bound are worst-case, so the experimental deviation
  probabilities and sample complexities are well below the theoretical values (Tasks 2, 5).
- **No-Free-Lunch.** With completely random targets, unseen labels are independent coin flips; no prediction
  rule beats any other, and the risk cannot drop below the error on the unseen part (Tasks 6–8). The learner
  simply cannot infer arbitrary unseen labels from the sample alone.
- **Inductive bias.** Restricting targets and hypotheses to thresholds injects prior knowledge. ERM then needs
  only to locate the threshold, so the risk falls to $0$ with few samples and the learner generalizes far
  beyond the observed inputs (Tasks 9–10) — the escape from No-Free-Lunch that assumptions provide.